# Score Fusion 앙상블

두 모델의 케이스x개체 유사도 행렬을 가중합해서 재랭킹:

```
sim_fused = w * sim_A + (1 - w) * sim_B
```

`w` 를 0~1로 스윕하면서, `compare_ensembles.ipynb` 에서 구한 **OR-coverage(이론적 상한)** 에
실제 score fusion이 얼마나 근접하는지 확인한다. 채점 로직(케이스 평균, gallery 캡, 개체단위 랭킹)은
`eval_case_level_*.py` 와 동일.

기본값은 실제 데이터에서 제일 좋았던 조합인 **CLIP-ReID + PetFace**, 데이터셋은 `shelter_hard_dogs`.

In [ ]:
import re
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset

ML_DIR = Path("..").resolve()  # 이 노트북은 ML/notebooks/ 에서 실행한다고 가정
CLIPREID_DIR = ML_DIR / "external" / "CLIP-ReID"
sys.path.insert(0, str(ML_DIR / "scripts"))   # arbase_model.py 재사용

FN_RE = re.compile(r"(\d+)_c(\d+)s(\d+)_(\d+)\.jpg$", re.I)
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def parse_pid(p):
    m = FN_RE.search(Path(p).name)
    return int(m.group(1)) if m else None


class ImgList(Dataset):
    def __init__(self, paths, tf):
        self.paths, self.tf = paths, tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        p = self.paths[i]
        return self.tf(Image.open(p).convert("RGB")), str(p)


def build_backbone(name, weight_path, clipreid_config):
    """eval-mode forward(imgs)->특징 을 내는 (forward_fn, transform) 리턴.
    eval_case_level_clipreid.py / eval_case_level_wildlife.py 의 모델 로딩부를 합친 것."""
    if name == "clipreid":
        sys.path.insert(0, str(CLIPREID_DIR))
        from config import cfg
        from model.make_model_clipreid import make_model

        cfg.merge_from_file(clipreid_config)
        cfg.freeze()
        tf = T.Compose([
            T.Resize(cfg.INPUT.SIZE_TEST),
            T.ToTensor(),
            T.Normalize(mean=cfg.INPUT.PIXEL_MEAN, std=cfg.INPUT.PIXEL_STD),
        ])
        model = make_model(cfg, num_class=95, camera_num=6, view_num=1)
        model.load_param(weight_path)
        model.to(DEVICE).eval()
        return (lambda imgs: model(imgs, cam_label=None, view_label=None)), tf

    ck = torch.load(weight_path, map_location=DEVICE, weights_only=False)
    if name == "megadescriptor":
        import timm
        model = timm.create_model("hf-hub:BVRA/MegaDescriptor-B-224", num_classes=0, pretrained=False)
        img_size = 224
    elif name == "petface":
        from torchvision.models import resnet50
        import torch.nn as nn
        model = resnet50(weights=None)
        model.fc = nn.Sequential(nn.Linear(model.fc.in_features, 512), nn.BatchNorm1d(512))
        img_size = 224
    elif name == "arbase":
        from arbase_model import ARBase
        model = ARBase(num_classes=ck["num_classes"], pretrained_backbone=False)
        img_size = 384
    else:
        raise ValueError(name)

    model.load_state_dict(ck["model"])
    model.to(DEVICE).eval()
    tf = T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    return (lambda imgs: model(imgs)), tf


@torch.no_grad()
def embed(forward_fn, paths, tf, batch=64):
    dl = DataLoader(ImgList(paths, tf), batch_size=batch, num_workers=0)
    feats, out_paths = [], []
    for imgs, ps in dl:
        f = F.normalize(forward_fn(imgs.to(DEVICE)), dim=1)
        feats.append(f.cpu())
        out_paths.extend(ps)
    return torch.cat(feats).numpy(), out_paths


def compute_id_similarity(case_pids, q_emb_raw, q_pid_raw, gallery_ids, id_index, g_emb, g_pid):
    """query 는 케이스(pid)별 평균, gallery 는 개체별 최댓값 -> [케이스 x 개체] 유사도 행렬."""
    case_emb = []
    for pid in case_pids:
        v = q_emb_raw[q_pid_raw == pid].mean(axis=0)
        case_emb.append(v / (np.linalg.norm(v) + 1e-12))
    case_emb = np.stack(case_emb)

    sim = case_emb @ g_emb.T
    id_sim = np.full((len(case_pids), len(gallery_ids)), -1.0, dtype=np.float32)
    for j, pid in enumerate(g_pid):
        col = id_index[pid]
        id_sim[:, col] = np.maximum(id_sim[:, col], sim[:, j])
    return id_sim


def recall_at_k(id_sim, case_pids, gallery_ids, ks):
    hits = {k: 0 for k in ks}
    for i, pid in enumerate(case_pids):
        order = np.argsort(-id_sim[i])
        ranked_ids = [gallery_ids[j] for j in order]
        rank = ranked_ids.index(pid) + 1 if pid in ranked_ids else None
        for k in ks:
            if rank is not None and rank <= k:
                hits[k] += 1
    n = len(case_pids)
    return {k: hits[k] / n for k in ks}


print("DEVICE:", DEVICE)

## 설정 — 조합/데이터셋 바꾸고 싶으면 여기만 수정

In [ ]:
# MODEL_A / MODEL_B 후보 (choices: "clipreid", "megadescriptor", "petface", "arbase")
# 각 모델의 체크포인트 경로:
#   clipreid       -> CLIPREID_DIR / "logs" / "mpdd_clipreid" / "ViT-B-16_60.pth"
#   megadescriptor -> ML_DIR / "checkpoints" / "megadescriptor_mpdd_best.pth"
#   petface        -> ML_DIR / "checkpoints" / "petface_mpdd_best.pth"
#   arbase         -> ML_DIR / "checkpoints" / "arbase_mpdd_best.pth"
MODEL_A, WEIGHT_A = "clipreid", str(CLIPREID_DIR / "logs" / "mpdd_clipreid" / "ViT-B-16_60.pth")
MODEL_B, WEIGHT_B = "petface", str(ML_DIR / "checkpoints" / "petface_mpdd_best.pth")
CLIPREID_CONFIG = str(CLIPREID_DIR / "configs" / "person" / "vit_clipreid_mpdd.yml")

# ROOT 후보 (데이터셋):
#   ML_DIR / "dataset" / "derived" / "MPDD_hard" / "MPDD" / "pytorch"          -> MPDD, 방해꾼만(열화 없음)
#   ML_DIR / "dataset" / "derived" / "MPDD_hard_corrupt" / "MPDD" / "pytorch"  -> MPDD, 방해꾼+query열화(시뮬레이션)
#   ML_DIR / "dataset" / "derived" / "shelter_hard"                            -> 진짜 보호소, 개+고양이 섞임
#   ML_DIR / "dataset" / "derived" / "shelter_hard_dogs"                       -> 진짜 보호소, 개만
ROOT = ML_DIR / "dataset" / "derived" / "shelter_hard_dogs"
GALLERY_PER_ID = 2
WEIGHTS_SWEEP = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]  # w=1.0 -> A만, w=0.0 -> B만 (검산용)

## 임베딩 추출 + 단독 성능

In [ ]:
q_paths_all = sorted((ROOT / "query").glob("*.jpg"))
g_paths_all = sorted((ROOT / "gallery").glob("*.jpg"))

by_pid = defaultdict(list)
for p in g_paths_all:
    by_pid[parse_pid(p)].append(p)
g_paths = [p for plist in by_pid.values() for p in plist[:GALLERY_PER_ID]]
gallery_ids = sorted(by_pid.keys())
id_index = {pid: i for i, pid in enumerate(gallery_ids)}
case_pids = sorted({parse_pid(p) for p in q_paths_all})

print(f"gallery {len(g_paths_all)}장 -> {GALLERY_PER_ID}장/개체 캡 후 {len(g_paths)}장 ({len(gallery_ids)}개체)")
print(f"query {len(q_paths_all)}장 -> {len(case_pids)}케이스")

id_sims = {}
solo_rows = {}
for tag, name, weight in [("A", MODEL_A, WEIGHT_A), ("B", MODEL_B, WEIGHT_B)]:
    forward_fn, tf = build_backbone(name, weight, CLIPREID_CONFIG)
    g_emb, g_used = embed(forward_fn, g_paths, tf)
    g_pid = np.array([parse_pid(p) for p in g_used])
    q_emb_raw, q_used = embed(forward_fn, q_paths_all, tf)
    q_pid_raw = np.array([parse_pid(p) for p in q_used])

    id_sim = compute_id_similarity(case_pids, q_emb_raw, q_pid_raw, gallery_ids, id_index, g_emb, g_pid)
    id_sims[tag] = id_sim
    solo_rows[f"{tag}={name}"] = recall_at_k(id_sim, case_pids, gallery_ids, [1, 5, 10])

solo_df = pd.DataFrame(solo_rows).T.rename(columns={1: "Recall@1", 5: "Recall@5", 10: "Recall@10"})
display(solo_df.style.format("{:.1%}"))

## 가중치 스윕

In [ ]:
rows = {}
for w in WEIGHTS_SWEEP:
    fused = w * id_sims["A"] + (1 - w) * id_sims["B"]
    rows[f"w={w:.1f}"] = recall_at_k(fused, case_pids, gallery_ids, [1, 5, 10])

sweep_df = pd.DataFrame(rows).T.rename(columns={1: "Recall@1", 5: "Recall@5", 10: "Recall@10"})
best_w = sweep_df["Recall@1"].idxmax()

display(
    sweep_df.style.format("{:.1%}")
    .background_gradient(cmap="Greens", axis=0)
    .apply(lambda s: ["font-weight: bold" if i == best_w else "" for i in s.index], axis=0)
)
print(f"최고 Recall@1: w={best_w} 에서 {sweep_df.loc[best_w, 'Recall@1']:.1%}")
print("(참고: compare_ensembles.ipynb 의 OR-coverage 는 '완벽한 심판'을 가정한 이론적 상한이라,")
print(" 여기 실제 score fusion 결과가 그보다 낮게 나오는 게 정상입니다.)")